In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df = pd.read_csv('credit_card_cleaned.csv')
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

print("="*80)
print("CREDIT CARD TRANSACTIONS - INSIGHTS GENERATION")
print("="*80)

In [ ]:
print("\n" + "="*80)
print("INSIGHT 1: FRAUD RISK ASSESSMENT")
print("="*80)

fraud_rate = df['is_fraud'].mean() * 100
total_fraud_loss = df[df['is_fraud']==1]['amt'].sum()
avg_fraud_amount = df[df['is_fraud']==1]['amt'].mean()

print(f"Overall Fraud Rate: {fraud_rate:.4f}%")
print(f"Total Financial Loss from Fraud: ${total_fraud_loss:,.2f}")
print(f"Average Fraud Transaction Amount: ${avg_fraud_amount:.2f}")
print(f"Total Fraudulent Transactions: {df['is_fraud'].sum():,} out of {len(df):,}")

if fraud_rate < 0.1:
    risk_level = "LOW"
elif fraud_rate < 0.5:
    risk_level = "MEDIUM"
elif fraud_rate < 1.0:
    risk_level = "HIGH"
else:
    risk_level = "CRITICAL"

print(f"\nRISK LEVEL: {risk_level}")
print(f"Insight: {fraud_rate:.4f}% of all transactions are fraudulent, resulting in ${total_fraud_loss:,.2f} total loss.")

In [ ]:
print("\n" + "="*80)
print("INSIGHT 2: HIGH-RISK PATTERNS")
print("="*80)

high_risk_hour = df.groupby('hour')['is_fraud'].mean().idxmax()
high_risk_hour_rate = df.groupby('hour')['is_fraud'].mean().max() * 100

print(f"Most Dangerous Hour: {high_risk_hour}:00 (Fraud rate: {high_risk_hour_rate:.2f}%)")

if 'day_of_week' in df.columns:
    high_risk_day = df.groupby('day_of_week')['is_fraud'].mean().idxmax()
    high_risk_day_rate = df.groupby('day_of_week')['is_fraud'].mean().max() * 100
    print(f"Most Dangerous Day: {high_risk_day} (Fraud rate: {high_risk_day_rate:.2f}%)")

high_amount_risk = df[df['amt'] > df['amt'].quantile(0.95)]['is_fraud'].mean() * 100
low_amount_risk = df[df['amt'] <= df['amt'].quantile(0.95)]['is_fraud'].mean() * 100

print(f"High Amount Transactions (>95th percentile) Fraud Rate: {high_amount_risk:.2f}%")
print(f"Normal Amount Transactions Fraud Rate: {low_amount_risk:.2f}%")
print(f"Insight: Transactions over ${df['amt'].quantile(0.95):.2f} are {high_amount_risk/low_amount_risk:.1f}x more likely to be fraudulent.")

In [ ]:
print("\n" + "="*80)
print("INSIGHT 3: CATEGORY RISK ANALYSIS")
print("="*80)

category_fraud = df.groupby('category')['is_fraud'].agg(['count', 'mean']).round(4)
category_fraud = category_fraud[category_fraud['count'] > 100].sort_values('mean', ascending=False)

print("TOP 5 HIGHEST RISK CATEGORIES:")
for i, (cat, row) in enumerate(category_fraud.head(5).iterrows(), 1):
    print(f"{i}. {cat}: {row['mean']*100:.2f}% fraud rate ({int(row['count'])} transactions)")

print("\nLOWEST RISK CATEGORIES:")
for i, (cat, row) in enumerate(category_fraud.tail(5).iterrows(), 1):
    print(f"{i}. {cat}: {row['mean']*100:.2f}% fraud rate ({int(row['count'])} transactions)")

top_risk_category = category_fraud.index[0]
top_risk_rate = category_fraud.iloc[0]['mean'] * 100
avg_risk_rate = df['is_fraud'].mean() * 100

print(f"\nInsight: {top_risk_category} category has {top_risk_rate/avg_risk_rate:.1f}x higher fraud rate than average.")

In [ ]:
print("\n" + "="*80)
print("INSIGHT 4: GEOGRAPHIC FRAUD HOTSPOTS")
print("="*80)

if 'state' in df.columns:
    state_fraud = df.groupby('state')['is_fraud'].agg(['count', 'mean']).round(4)
    state_fraud = state_fraud[state_fraud['count'] > 500].sort_values('mean', ascending=False)
    
    print("TOP 5 STATES WITH HIGHEST FRAUD RATE:")
    for i, (state, row) in enumerate(state_fraud.head(5).iterrows(), 1):
        print(f"{i}. {state}: {row['mean']*100:.2f}% fraud rate ({int(row['count'])} transactions)")
    
    print("\nSAFEST STATES (LOWEST FRAUD RATE):")
    for i, (state, row) in enumerate(state_fraud.tail(5).iterrows(), 1):
        print(f"{i}. {state}: {row['mean']*100:.2f}% fraud rate ({int(row['count'])} transactions)")
    
    highest_fraud_state = state_fraud.index[0]
    lowest_fraud_state = state_fraud.index[-1]
    print(f"\nInsight: Customers in {highest_fraud_state} are {state_fraud.iloc[0]['mean']/state_fraud.iloc[-1]['mean']:.1f}x more likely to experience fraud than those in {lowest_fraud_state}.")

In [ ]:
print("\n" + "="*80)
print("INSIGHT 5: CUSTOMER RISK SEGMENTATION")
print("="*80)

customer_risk = df.groupby('cc_num').agg({
    'is_fraud': ['sum', 'mean'],
    'amt': ['count', 'mean']
}).round(4)

customer_risk.columns = ['fraud_count', 'fraud_rate', 'transaction_count', 'avg_amount']
customer_risk = customer_risk[customer_risk['transaction_count'] > 10]

high_risk_customers = customer_risk[customer_risk['fraud_rate'] > 0.10]
medium_risk_customers = customer_risk[(customer_risk['fraud_rate'] > 0.02) & (customer_risk['fraud_rate'] <= 0.10)]
low_risk_customers = customer_risk[customer_risk['fraud_rate'] <= 0.02]

print(f"High Risk Customers (>10% fraud rate): {len(high_risk_customers)} customers")
print(f"Medium Risk Customers (2-10% fraud rate): {len(medium_risk_customers)} customers")
print(f"Low Risk Customers (<2% fraud rate): {len(low_risk_customers)} customers")

if len(high_risk_customers) > 0:
    print(f"\nHigh Risk Customer Profile:")
    print(f"  - Average transactions: {high_risk_customers['transaction_count'].mean():.0f}")
    print(f"  - Average amount per transaction: ${high_risk_customers['avg_amount'].mean():.2f}")

print(f"\nInsight: {len(high_risk_customers)} customers account for a disproportionate amount of fraud risk and should be prioritized for monitoring.")

In [ ]:
print("\n" + "="*80)
print("INSIGHT 6: TEMPORAL FRAUD PATTERNS")
print("="*80)

df['hour'] = df['trans_date_trans_time'].dt.hour
df['is_weekend'] = df['trans_date_trans_time'].dt.dayofweek >= 5

hourly_fraud = df.groupby('hour')['is_fraud'].mean()
weekend_fraud_rate = df[df['is_weekend']]['is_fraud'].mean() * 100
weekday_fraud_rate = df[~df['is_weekend']]['is_fraud'].mean() * 100

print(f"Weekend Fraud Rate: {weekend_fraud_rate:.4f}%")
print(f"Weekday Fraud Rate: {weekday_fraud_rate:.4f}%")

if weekend_fraud_rate > weekday_fraud_rate:
    print(f"Insight: Weekend transactions are {(weekend_fraud_rate/weekday_fraud_rate):.1f}x riskier than weekday transactions.")
else:
    print(f"Insight: Weekday transactions are {(weekday_fraud_rate/weekend_fraud_rate):.1f}x riskier than weekend transactions.")

peak_fraud_hours = hourly_fraud.nlargest(3)
print(f"\nPeak Fraud Hours: {', '.join([f'{h}:00 ({rate*100:.2f}%)' for h, rate in peak_fraud_hours.items()])}")
print(f"Insight: Most fraud occurs between {peak_fraud_hours.index[0]}:00 and {peak_fraud_hours.index[0]+1}:00.")

In [ ]:
print("\n" + "="*80)
print("INSIGHT 7: FINANCIAL IMPACT ANALYSIS")
print("="*80)

fraud_by_category = df[df['is_fraud']==1].groupby('category')['amt'].sum().sort_values(ascending=False)
total_fraud = fraud_by_category.sum()

print("TOP 5 CATEGORIES BY FRAUD LOSS AMOUNT:")
for i, (cat, amount) in enumerate(fraud_by_category.head(5).items(), 1):
    percentage = (amount / total_fraud) * 100
    print(f"{i}. {cat}: ${amount:,.2f} ({percentage:.1f}% of total fraud loss)")

avg_fraud_amount_by_cat = df[df['is_fraud']==1].groupby('category')['amt'].mean().sort_values(ascending=False)
print("\nCATEGORIES WITH HIGHEST AVERAGE FRAUD AMOUNT:")
for i, (cat, amount) in enumerate(avg_fraud_amount_by_cat.head(3).items(), 1):
    print(f"{i}. {cat}: ${amount:.2f} average fraud transaction")

print(f"\nInsight: {fraud_by_category.index[0]} category suffers the highest financial loss from fraud, accounting for {(fraud_by_category.iloc[0]/total_fraud)*100:.1f}% of total fraud losses.")

In [ ]:
print("\n" + "="*80)
print("INSIGHT 8: PREVENTION RECOMMENDATIONS")
print("="*80)

print("RECOMMENDATION 1: Time-Based Monitoring")
print(f"  - Increase monitoring during {high_risk_hour}:00 - {(high_risk_hour+2)%24}:00")
print(f"  - Fraud rate is {high_risk_hour_rate:.2f}% during this window")

print("\nRECOMMENDATION 2: Category-Specific Rules")
print(f"  - Implement additional verification for {top_risk_category} transactions")
print(f"  - This category has {top_risk_rate/avg_risk_rate:.1f}x higher fraud rate")

print("\nRECOMMENDATION 3: Amount Threshold Alerts")
print(f"  - Flag all transactions > ${df['amt'].quantile(0.95):.2f} for review")
print(f"  - High amount transactions are {high_amount_risk/low_amount_risk:.1f}x more likely fraudulent")

print("\nRECOMMENDATION 4: Geographic Restrictions")
if 'state' in df.columns:
    print(f"  - Enhance verification for transactions originating from {highest_fraud_state}")
    print(f"  - Consider velocity checks for cross-state transactions")

print("\nRECOMMENDATION 5: Customer Education")
print(f"  - Notify customers about {len(high_risk_customers)} high-risk accounts")
print(f"  - Implement SMS alerts for transactions in high-risk categories")

In [ ]:
print("\n" + "="*80)
print("INSIGHT 9: KEY PERFORMANCE INDICATORS")
print("="*80)

kpi_data = {
    'Metric': [
        'Fraud Detection Rate',
        'False Positive Rate (estimated)',
        'Average Fraud Loss per Incident',
        'Customer Fraud Exposure Rate',
        'High Risk Transaction Percentage'
    ],
    'Value': [
        f"{fraud_rate:.4f}%",
        f"{(df['amount_outlier'].sum() - df[df['amount_outlier']==True]['is_fraud'].sum()) / len(df) * 100:.2f}%",
        f"${avg_fraud_amount:.2f}",
        f"{(len(high_risk_customers)/len(df['cc_num'].unique())*100):.1f}%",
        f"{len(df[df['amt'] > df['amt'].quantile(0.95)])/len(df)*100:.1f}%"
    ]
}

kpi_df = pd.DataFrame(kpi_data)
print(kpi_df.to_string(index=False))

In [ ]:
print("\n" + "="*80)
print("INSIGHT 10: EXECUTIVE SUMMARY")
print("="*80)

print(f"""
FRAUD OVERVIEW:
- Total transactions analyzed: {len(df):,}
- Fraudulent transactions: {df['is_fraud'].sum():,} ({fraud_rate:.4f}%)
- Total fraud loss: ${total_fraud_loss:,.2f}

TOP 3 RISK FACTORS:
1. Category: {top_risk_category} ({top_risk_rate:.2f}% fraud rate)
2. Hour: {high_risk_hour}:00 ({high_risk_hour_rate:.2f}% fraud rate)
3. Amount: >${df['amt'].quantile(0.95):.2f} ({high_amount_risk:.2f}% fraud rate)

PRIORITY ACTIONS:
- Implement real-time monitoring for {top_risk_category} transactions
- Flag transactions between {high_risk_hour}:00-{(high_risk_hour+2)%24}:00
- Set amount threshold alerts at ${df['amt'].quantile(0.95):.2f}
- Review {len(high_risk_customers)} high-risk customer accounts
""")

# Save insights to file
insights_summary = {
    'fraud_rate': fraud_rate,
    'total_fraud_loss': total_fraud_loss,
    'avg_fraud_amount': avg_fraud_amount,
    'high_risk_category': top_risk_category,
    'high_risk_hour': high_risk_hour,
    'high_risk_amount_threshold': df['amt'].quantile(0.95),
    'high_risk_customers_count': len(high_risk_customers) if 'high_risk_customers' in locals() else 0
}

pd.DataFrame([insights_summary]).to_csv('insights_summary.csv', index=False)
print("\nInsights saved to 'insights_summary.csv'")